# ExaHyPE 2 Tutorials - Part 1: The Acoustic Equations

Welcome to the first ExaHyPE 2 tutorial. ExaHyPE 2 is an engine for **hyperbolic partial differential equations** - PDEs with a finite propagation speed, such as sound, elastic or electromagnetic waves. You describe your PDE in a few Python lines and a small C++ file, and the engine generates an optimised, parallel solver for you.

In this part you solve the **acoustic equations** driven by a **point source**: a short pulse fired near the **bottom-left corner** of the box sends out an expanding circular sound wave. You will learn the complete workflow on a fully worked example:

1. import the ExaHyPE 2 Python API,
2. define a **project** (the computational domain and how it is discretised),
3. define a **solver** for your PDE,
4. build, run and **visualise** the result.

The accompanying C++ file **`ADERDGSolver.cpp`** holds the implementation of the PDE terms; open it alongside this notebook.

## The Acoustic Equations

We solve the linear acoustic equations in two dimensions for the pressure $p$ and the velocity $\mathbf{v} = (v_1, v_2)$:

$$
\frac{\partial}{\partial t}
\begin{pmatrix} p \\ v_1 \\ v_2 \end{pmatrix}
+ \nabla \cdot
\begin{pmatrix} K_0 v_1 & K_0 v_2 \\ p/\rho & 0 \\ 0 & p/\rho \end{pmatrix}
= \begin{pmatrix} s(\mathbf{x}, t) \\ 0 \\ 0 \end{pmatrix}.
$$

Here $\rho = 1$ is the density and $K_0 = \rho c^2 = 1$ the bulk modulus, so the waves travel at the speed of sound $c = \sqrt{K_0/\rho} = 1$ (this is also the largest eigenvalue of the flux, which ExaHyPE needs for the Riemann solver and the time step). The right-hand side $s(\mathbf{x}, t)$ is the **point source**: a Gaussian pulse in time injected at a single point near the bottom-left corner of the domain.

## Importing the API

ExaHyPE 2 is a Python-driven code generator. Everything starts with a single import.

In [ ]:
import sys, os

# Make the ExaHyPE 2 Python API importable. Inside the tutorial Docker image
# this is already on PYTHONPATH; the line below also makes the notebook work
# from a plain checkout.
sys.path.insert(0, os.path.abspath("../../api/"))

import exahype2

## Defining the Project

The **project** describes *where* and *how* we solve, independently of the PDE: the generated C++ namespace, the executable name, the spatial domain (`size`/`offset`) and the boundary type. We use a $2 \times 2$ box centred on the origin with **non-periodic** boundaries, so the expanding wave reflects when it reaches the walls.

In [ ]:
my_project = exahype2.Project(
    namespace    = ["tutorials", "exahype2", "acoustic"],
    project_name = "Acoustic",
    executable   = "Acoustic",
    dimensions   = 2,
    size         = [ 2.0,  2.0],
    offset       = [-1.0, -1.0],
    periodic_bc  = [False, False],   # non-periodic: waves reflect off the walls
)

## Defining the Solver

We use **ADER-DG** (a high-order discontinuous Galerkin method with ADER time stepping). The solver is named `ADERDGSolver` - this name determines the generated C++ class and the output file prefix, so remember it for the visualisation step.

* `order` is the polynomial degree per cell (5 gives very high spatial accuracy);
* `time_step_relaxation` is a safety factor (< 1) on the stable time-step size;
* `unknowns` names the evolved quantities; `v` has multiplicity 2 (the two velocity components), so we can address them by name in C++.

In [ ]:
aderdg_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                 = "ADERDGSolver",
    order                = 5,
    time_step_relaxation = 0.9,
    unknowns             = {"p": 1, "v": 2},
    auxiliary_variables  = {},
)

Next we declare which PDE terms we implement. Besides the **flux** and the **maximum eigenvalue**, this scenario uses ExaHyPE's dedicated **point-source API** (`number_of_point_sources`, `point_source`, `init_point_source_location`); we also give the **initial** and **boundary** conditions. `User_Defined_Implementation` means we write each one by hand in `ADERDGSolver.cpp`. Because the system is linear we switch on the optimised linear ADER-DG kernels - which the point-source API also requires.

In [ ]:
aderdg_solver.set_implementation(
    initial_conditions         = exahype2.solvers.PDETerms.User_Defined_Implementation,
    boundary_conditions        = exahype2.solvers.PDETerms.User_Defined_Implementation,
    flux                       = exahype2.solvers.PDETerms.User_Defined_Implementation,
    max_eigenvalue             = exahype2.solvers.PDETerms.User_Defined_Implementation,
    number_of_point_sources    = 1,
    point_source               = exahype2.solvers.PDETerms.User_Defined_Implementation,
    init_point_source_location = exahype2.solvers.PDETerms.User_Defined_Implementation,
)

# The system is linear, so switch on the optimised linear ADER-DG kernels (also a
# prerequisite for the dedicated point-source API).
aderdg_solver.add_kernel_optimisations(is_linear=True)

my_project.add_solver(aderdg_solver)

## Building the Project

We register our hand-written implementation file and then call `generate_and_build()`, which turns the ExaHyPE description into C++ and compiles everything. The first build can take a minute.

In [ ]:
my_project.output.makefile.add_cpp_file("ADERDGSolver.cpp")
my_project.generate_and_build()

## The Implementation File

Open **`ADERDGSolver.cpp`**. For this worked example it is already complete; each function mirrors the maths above:

* `maxEigenvalue` returns the sound speed $c$;
* `flux` fills the flux vector `F` for the requested `normal` direction, using the named components `Q[Shortcuts::p]`, `Q[Shortcuts::v + 0/1]`;
* `initialCondition` sets a quiescent (zero) state - the waves come entirely from the source;
* `boundaryConditions` prescribes a zero exterior state, off which waves reflect (the domain is non-periodic, so this *is* called);
* `pointSource` and `initPointSourceLocations` are the **point source**: a Gaussian pulse in time injected at the single source location near the bottom-left corner.


## Running the Simulation

We run the freshly built executable. The runtime flags control the simulation *without* recompiling:

* `-et` end time;
* `-ns` number of output snapshots (drives the animation below);
* `-l` the uniform mesh level: level $\ell$ covers the domain with $k^\ell$ cells per direction, k = 3 on the Peano curve, 2 on the Hilbert curve.

We also set `OMP_NUM_THREADS` to run on four OpenMP threads.

In [ ]:
import os, subprocess

# Runtime parameters - edit these freely, no rebuild needed.
end_time  = 2.2     # -et : physical end time (long enough for the wave to sweep to the upper-right)
snapshots = 22      # -ns : number of output snapshots (drives the animation)
level     = 3       # -l  : uniform mesh level (k^level cells per direction, k = 3 on the Peano curve, 2 on the Hilbert curve)

# Run with 4 OpenMP threads (the notebook equivalent of: export OMP_NUM_THREADS=4)
os.environ["OMP_NUM_THREADS"] = "4"

subprocess.run(
    ["./Acoustic", "-et", str(end_time), "-ns", str(snapshots), "-l", str(level)],
    check=True,
)

## Visualising the Result

`render_solution.show_solution` renders the snapshots and gives you a time slider, a field selector and a play button. On opening (and when you switch fields) all frames are rendered in parallel across cores and cached - a brief *Prefetching N/M* - so scrubbing and playback are then smooth. Drag the slider to watch the circular wave expand from the source and reflect off the walls. Use the field dropdown to switch between `p` and `v`.

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath("../../"))
from postprocessing import render_solution
importlib.reload(render_solution)
render_solution.show_solution("ADERDGSolver", field="p")

## Experiment

A couple of things you can change here and re-run with **no rebuild**:

* pass `field="v"` to `show_solution` to view a velocity component instead of the pressure;
* raise `level` to 4, or `end_time` and `snapshots`, in the run cell and re-run.

For the full list of experiments (including moving the source, which needs a rebuild) see the documentation page, *Part 1 - The Acoustic Equations*. In **Part 2 - The Shallow Water Equations** you will implement a non-linear solver yourself - and meet static mesh refinement.